# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to load and analyze the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

#### Dataset Source
The FAIR² dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) available via a schema URL.

Dataset DOI: [10.71728/senscience.qs2f-h81p](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading

Load the FAIR² dataset's metadata and prepare to access its record sets using `mlcroissant`. This step will print the dataset title and its description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset's Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Remains a mlcroissant.metadata.Dataset object

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

This section lists available record sets and their fields in the dataset. **All references are by their `@id` fields.**

**Record Set(s) present:**

In [ ]:
# List all record sets and their field @ids, following Croissant schema conventions.

record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in Croissant metadata.")
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id']}")
        if 'field' in rs:
            # 'field' can be list of field dicts or single field
            if isinstance(rs['field'], list):
                for f in rs['field']:
                    if isinstance(f, dict):
                        print(f"    field @id: {f.get('@id', str(f))}")
                    else:
                        print(f"    field @id: {str(f)}")
            elif isinstance(rs['field'], dict):
                print(f"    field @id: {rs['field'].get('@id', str(rs['field']))}")
            else:
                print(f"    field @id: {str(rs['field'])}")
        else:
            print("    No fields described for this record set.")

For this dataset, the primary tabular data is typically in a single record set. Let's print some sample records from the first available record set (by its `@id`).

In [ ]:
# Print several records from the first record set (@id based)

record_sets = dataset.record_sets
if not record_sets:
    print("No record sets available.")
else:
    main_record_set_id = record_sets[0]['@id']
    print(f"First Record Set @id: {main_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
        print(record)
        if i >= 2:
            break  # Show just a few for illustration

## 3. Data Extraction

Load all records from the main record set (using its `@id`) into a [Pandas](https://pandas.pydata.org/) DataFrame for downstream analysis.

We'll extract and inspect the columns and first few rows.

In [ ]:
# Extract all records from all record sets into DataFrames by their `@id`
dataframes = {}
for rs in dataset.record_sets:
    rs_id = rs['@id']
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# For exploration, we'll use the main record set, already found previously
print(f"Columns in main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
print("\nFirst rows:")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

We'll select a numeric field (referenced by its field `@id`) for demonstration. You'll see filtering, normalization, and grouping by a categorical field—always using `@id`.

> Note: Replace `<numeric_field_id>` and `<group_field_id>` with known `@id` values from earlier output, e.g., `'cr:age'` for age if present.

In [ ]:
# --- Specify fields by @id as found in metadata or printout ---
# You may need to print(list(dataframes[main_record_set_id].columns)) to choose appropriately

# For this example, we assume there are fields 'cr:age' and 'cr:sex' (column names by their @id)
df = dataframes[main_record_set_id]
print("Available columns (@id):", df.columns.tolist())

# Use actual field @id present in this dataset
numeric_field = None
group_field = None
for col in df.columns:
    # Heuristics for demonstration, adjust if needed
    if 'age' in col.lower():
        numeric_field = col
    if ('sex' in col.lower()) or ('gender' in col.lower()):
        group_field = col
if not numeric_field:
    # else pick first numeric column
    for col in df.select_dtypes(include='number').columns:
        numeric_field = col
        break
if not group_field:
    for col in df.columns:
        if 'site' in col.lower() or 'location' in col.lower():
            group_field = col
            break
print(f"Selected numeric field for EDA: {numeric_field}")
print(f"Selected group field for grouping: {group_field}")

if numeric_field:
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())
    normalized_col = f"{numeric_field}_normalized"
    filtered_df[normalized_col] = (
        pd.to_numeric(filtered_df[numeric_field], errors='coerce') - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, normalized_col]].head())
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field} by {group_field}:")
        print(grouped_df)
else:
    print("No suitable numeric field available for analysis in this dataset.")

## 5. Visualization

Let's visualize the selected numeric field and its grouping. (You can select another field by setting `numeric_field` and `group_field` by their `@id`).

In [ ]:
import matplotlib.pyplot as plt

if numeric_field:
    plt.figure(figsize=(7, 4))
    df[numeric_field].hist(bins=15)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    if group_field and group_field in df.columns:
        means = df.groupby(group_field)[numeric_field].mean()
        means.plot(kind='bar')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric field was available for visualization.")

## 6. Conclusion

This notebook demonstrated how to discover, extract, and analyze the FAIR² dataset using [`mlcroissant`](https://github.com/mlcommons/croissant) and Python analytics tools. 

- **All fields and record sets were referenced by their `@id` as per Croissant best practices.**
- We demonstrated EDA steps (filtering, normalization, grouping) using the schema structure.
- This workflow supports reproducible research on clinical datasets described in the FAIR format.

For deeper analysis, refer to the record set/field documentation in the Croissant metadata:

```python
# View all metadata for a field by @id:
field_id = numeric_field
for rs in dataset.record_sets:
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for f in fields:
            if isinstance(f, dict) and f.get('@id') == field_id:
                pprint(f)
```